# 04 — Avaliação do Modelo

**Objetivo:** Avaliar o modelo treinado no conjunto de teste com métricas completas.

Neste notebook:
- Carregamos o melhor checkpoint salvo
- Calculamos acurácia, precisão, recall, F1 e AUC-ROC por classe
- Geramos matriz de confusão e curvas ROC
- Visualizamos predições corretas e erradas para análise de erros
- Aplicamos Grad-CAM para explicabilidade (XAI) e validação visual das ativações

## 1. Configuração

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from config import (
    CAMINHO_MODELOS, NOME_ARQUIVO_MODELO, ARQUITETURA_MODELO,
    NOMES_CLASSES, SEMENTE_ALEATORIA,
    MEDIA_IMAGENET, DESVIO_IMAGENET,
)
from dataset import criar_dataloaders
from modelo import criar_modelo
from avaliacao import (
    coletar_predicoes,
    calcular_metricas,
    plotar_matriz_confusao,
    plotar_curvas_roc,
)

torch.manual_seed(SEMENTE_ALEATORIA)
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DISPOSITIVO}')

## 2. Carregando o Modelo Treinado

In [ ]:
caminho_checkpoint = CAMINHO_MODELOS / NOME_ARQUIVO_MODELO

if not caminho_checkpoint.exists():
    raise FileNotFoundError(
        f'Checkpoint não encontrado: {caminho_checkpoint}\n'
        f'Execute o notebook 03_treinamento.ipynb primeiro.'
    )

# Recria a arquitetura e carrega os pesos
modelo = criar_modelo(arquitetura=ARQUITETURA_MODELO, congelar_backbone=False, dispositivo=DISPOSITIVO)
modelo.load_state_dict(torch.load(caminho_checkpoint, map_location=DISPOSITIVO))
modelo.eval()

print(f'Modelo carregado: {caminho_checkpoint.name}')

## 3. Carregando o Conjunto de Teste

In [ ]:
_, _, loader_teste = criar_dataloaders(num_workers=0)
print(f'Batches de teste: {len(loader_teste)}')
print(f'Total de imagens: {len(loader_teste.dataset)}')

## 4. Coletando Predições

In [ ]:
rotulos_reais, predicoes, probabilidades = coletar_predicoes(
    modelo=modelo,
    loader=loader_teste,
    dispositivo=DISPOSITIVO,
)

print(f'Amostras avaliadas : {len(rotulos_reais)}')
print(f'Shape probabilidades: {probabilidades.shape}')

## 5. Métricas de Classificação

In [ ]:
metricas = calcular_metricas(rotulos_reais, predicoes, probabilidades)

## 6. Matriz de Confusão

In [ ]:
plotar_matriz_confusao(rotulos_reais, predicoes, salvar=True)

## 7. Curvas ROC

In [ ]:
plotar_curvas_roc(rotulos_reais, probabilidades, salvar=True)

## 8. Análise de Erros — Visualizando Predições Incorretas

In [ ]:
def desnormalizar(tensor: torch.Tensor) -> np.ndarray:
    """Reverte a normalização ImageNet para exibição."""
    media  = torch.tensor(MEDIA_IMAGENET).view(3, 1, 1)
    desvio = torch.tensor(DESVIO_IMAGENET).view(3, 1, 1)
    img = tensor * desvio + media
    return np.clip(img.permute(1, 2, 0).numpy(), 0, 1)


def visualizar_erros(dataset, rotulos_reais, predicoes, probabilidades, n: int = 12):
    """Exibe imagens onde o modelo errou, com rótulo real e predito."""
    indices_erros = np.where(rotulos_reais != predicoes)[0]
    print(f'Total de erros: {len(indices_erros)} / {len(rotulos_reais)}')

    amostra = indices_erros[:n]
    cols = 4
    rows = (len(amostra) + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.2))
    axes = axes.flatten()

    for i, idx in enumerate(amostra):
        tensor_img, _ = dataset[idx]
        img = desnormalizar(tensor_img)

        real   = NOMES_CLASSES[rotulos_reais[idx]].split(' ')[0]
        predito = NOMES_CLASSES[predicoes[idx]].split(' ')[0]
        confianca = probabilidades[idx, predicoes[idx]]

        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(
            f'Real: {real}\nPredito: {predito} ({confianca:.1%})',
            fontsize=8, color='red'
        )

    # Oculta eixos vazios
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Exemplos de Predições Incorretas', fontsize=12)
    plt.tight_layout()
    plt.savefig('../reports/erros_predicao.png', dpi=120, bbox_inches='tight')
    plt.show()


visualizar_erros(loader_teste.dataset, rotulos_reais, predicoes, probabilidades)

## 9. Exemplos de Predições Corretas

In [ ]:
def visualizar_acertos(dataset, rotulos_reais, predicoes, probabilidades, n: int = 8):
    """Exibe imagens corretamente classificadas com alta confiança."""
    indices_acertos = np.where(rotulos_reais == predicoes)[0]
    # Ordena por confiança decrescente
    confiancas = probabilidades[indices_acertos, predicoes[indices_acertos]]
    indices_top = indices_acertos[np.argsort(confiancas)[::-1][:n]]

    cols = 4
    rows = (len(indices_top) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.2))
    axes = axes.flatten()

    for i, idx in enumerate(indices_top):
        tensor_img, _ = dataset[idx]
        img = desnormalizar(tensor_img)
        classe   = NOMES_CLASSES[rotulos_reais[idx]].split(' ')[0]
        conf     = probabilidades[idx, predicoes[idx]]

        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(f'{classe}\n{conf:.1%}', fontsize=8, color='green')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Predições Corretas (Maior Confiança)', fontsize=12)
    plt.tight_layout()
    plt.savefig('../reports/acertos_predicao.png', dpi=120, bbox_inches='tight')
    plt.show()


visualizar_acertos(loader_teste.dataset, rotulos_reais, predicoes, probabilidades)

## 10. Explicabilidade — Grad-CAM

Grad-CAM gera um mapa de calor sobre a tomografia mostrando quais regiões influenciaram a decisão do modelo.  
Isso permite verificar se o modelo está focando nas áreas pulmonares corretas ou em artefatos irrelevantes.

In [ ]:
from gradcam import GradCAM, visualizar_gradcam, inferencia_com_gradcam

# --- Grade: amostra mista do conjunto de teste (acertos e erros) ---
imagens_batch, rotulos_batch = next(iter(loader_teste))

visualizar_gradcam(
    modelo=modelo,
    imagens=imagens_batch,
    rotulos_reais=rotulos_batch.tolist(),
    dispositivo=DISPOSITIVO,
    n_imagens=8,
    salvar=True,
    nome_arquivo='gradcam_batch_teste.png',
)

## 11. Salvando Relatório Final

## 10. Salvando Relatório Final

In [ ]:
import json
from config import CAMINHO_REPORTS

relatorio_final = {
    'arquitetura': ARQUITETURA_MODELO,
    'acuracia_teste': float(metricas['acuracia']),
    'auc_roc_macro': float(metricas['auc_roc']),
    'n_amostras_teste': int(len(rotulos_reais)),
    'n_erros': int((rotulos_reais != predicoes).sum()),
}

with open(CAMINHO_REPORTS / 'resultado_final.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio_final, f, indent=2, ensure_ascii=False)

print('Relatório salvo em reports/resultado_final.json')
print('\nResumo Final:')
for k, v in relatorio_final.items():
    print(f'  {k}: {v}')